# RealPDE — is the default ±5% SPS interval band too narrow? (GPU sweep)

The SPS scorer's **default** interval is `lower = pred - 0.05*|pred|`, `upper = pred + 0.05*|pred|` (total width `0.1*|pred|`). This notebook sweeps a fixed **relative** band `pred ± w*|pred|` over a set of real data trajectories with the **frozen CNO** baseline and reports the official SPS score + coverage for each width, so we can see whether widening to 10%/20%/... actually buys SPS.

Because the prediction is fixed and only the interval band changes, the differences are **purely** the sharpness-vs-coverage tradeoff of the interval width — exactly what we want to isolate.

In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR} ]; then echo "Pulling {REPO_DIR}..."; cd {REPO_DIR} && git pull; else echo "Cloning {REPO_URL}..."; git clone {REPO_URL} {REPO_DIR}; fi

In [ ]:
%cd {REPO_DIR}
import sys, torch
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}, device', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('GPU is off here — run this notebook with a GPU accelerator for speed.') if device=='cpu' else None

## 1. Resolve real data

Prefer an attached **real `test_real/`** split (same layout the leaderboard uses, with the official `mean_std_real.pt`). If only `train_real/` is attached, stage `N` trajectories to `/kaggle/working/real30` with protocol-exact stats via `scripts/stage_real30.py`.

In [ ]:
from pathlib import Path
REPO = Path(REPO_DIR)
DATA_ROOT = next((c for c in ['/kaggle/input/datasets/nthday/realpde', '/kaggle/input/realpde'] if Path(c).exists()), None)
print(f'DATA_ROOT -> {DATA_ROOT}')

# Case A: real test split present (best).
DATA_DIR = None
cands = [d for d in ([Path(DATA_ROOT)] if DATA_ROOT else []) for d in [d]]
for root in ([Path(DATA_ROOT)] if DATA_ROOT else []):
    for sub in ['test_real', 'test']:
        tdir = root / sub
        stats = root / 'mean_std_real.pt'
        if tdir.is_dir() and any(tdir.glob('*.h5')) and stats.exists():
            DATA_DIR = root
            break
if DATA_DIR is None:
    print('[info] no attached REAL test split with stats; will stage 30 from train_real.')
else:
    print(f'[ok] USING REAL test split at {DATA_DIR} (official mean_std_real.pt)')

In [ ]:
# Stage from train_real only when no real test split is available.
import subprocess, sys, shutil
STAGE_DIR = Path('/kaggle/working/real30')
N_FILES, N_FRAMES, SEED = 30, 200, 42
data_dir_path = DATA_DIR

if DATA_DIR is None:
    train_real = None
    if DATA_ROOT is not None:
        base = Path(DATA_ROOT) / 'train_real'
        cands = [base] if base.is_dir() else []
        if not any(c for c in cands if any(c.glob('*.h5'))):
            from collections import Counter
            cands = [Counter(p.parent for p in base.rglob('*.h5')).most_common(1)[0][0] if list(base.rglob('*.h5')) else None]
        train_real = next((c for c in cands if c and any(c.glob('*.h5'))), None)
    print(f'train_real -> {train_real}')
    if train_real is not None:
        if STAGE_DIR.exists():
            shutil.rmtree(STAGE_DIR)
        r = subprocess.run([sys.executable, 'scripts/stage_real30.py', '--src', str(train_real),
                            '--dst', str(STAGE_DIR), '--n-files', str(N_FILES),
                            '--frames', str(N_FRAMES), '--seed', str(SEED)], cwd=REPO)
        data_dir_path = STAGE_DIR
        print(f'[stage] exit code: {r.returncode}')
    else:
        print('[FAIL] no real data at all — attach the realpde dataset (train_real/ or test/).')

print(f'\nFINAL DATA_DIR = {data_dir_path}')
assert data_dir_path is not None and (Path(data_dir_path) / 'test_real').is_dir(), 'need test_real/ + mean_std_real.pt'

## 2. Load the frozen CNO baseline + official normalizer

We drive `submission_v5`'s frozen-forecast path (which loads `model.pth` via `load_baseline`) to get real CNO predictions in normalized space, then denormalize for the width sweep.

In [ ]:
# Stage the real CNO checkpoint as the v5 model so get_ttt_model loads it.
import shutil
sub_dir = REPO / 'submissions' / 'submission_v5'
main = sub_dir / 'model.pth'
if not main.exists():
    roots = [Path(DATA_ROOT), Path('/kaggle/input/realpde'), Path('/kaggle/working/checkpoints')]
    ckpt = None
    for r in roots:
        if r is None or not r.exists():
            continue
        cands = sorted([p for p in r.rglob('*.pth') if 'cno' in p.name.lower()],
                       key=lambda p: (0 if 'sim_real' in p.name.lower() else 1, p.name))
        if cands:
            ckpt = cands[0]
            break
    if ckpt:
        shutil.copy(ckpt, main)
    print(f'staged model.pth <- {ckpt}')
else:
    print('model.pth already staged')

In [ ]:
sys.path.insert(0, str(sub_dir))
import submission as v5
mod = v5.get_ttt_model(str(sub_dir), device)
print('model ready on', device)

In [ ]:
import h5py, numpy as np, torch, os
IN_STEP, OUT_STEP, INTERVAL, SUB_S = 20, 20, 20, 2
HORIZON = IN_STEP + OUT_STEP

# Official normalizer (test split) — protocol-exact.
stats = Path(data_dir_path) / 'mean_std_real.pt'
mi, mt, si, st = torch.load(stats, map_location='cpu', weights_only=False)
one = torch.ones_like
norm = dict(mean_in=mi.float(), mean_tgt=mt.float(),
            std_in=torch.where(si == 0, one(si), si).float(),
            std_tgt=torch.where(st == 0, one(st), st).float())

def denorm(p):  # p normalized -> physical units (u,v only graded)
    c = p.shape[-1]
    return p * norm['std_tgt'][..., :c] + norm['mean_tgt'][..., :c]

def build_stream(base):
    tdir = Path(base) / 'test_real'
    files = sorted(f for f in os.listdir(tdir) if f.endswith('.h5'))
    entries, frames_of = [], {}
    for name in files:
        with h5py.File(tdir / name, 'r') as f:
            frames_of[name] = f['u'].shape[0]
        for t in range(0, frames_of[name] - HORIZON + 1, INTERVAL):
            entries.append((name, t))
    entries.sort(key=lambda e: (e[0], e[1]))
    stream, prev = [], None
    for name, tid in entries:
        with h5py.File(tdir / name, 'r') as f:
            end = min(tid + HORIZON, frames_of[name])
            u = f['u'][tid:end, ::SUB_S, ::SUB_S]
            v = f['v'][tid:end, ::SUB_S, ::SUB_S]
        p = np.zeros_like(u)
        data = np.stack([u, v, p], axis=-1)
        if data.shape[0] < HORIZON:
            data = np.concatenate([data, np.repeat(data[-1:], HORIZON - data.shape[0], 0)], 0)
        stream.append({'input': data[:IN_STEP], 'target': data[IN_STEP:HORIZON], 'is_first': name != prev})
        prev = name
    return stream

def preprocess(x, y):
    c1, c2 = x.shape[-1], y.shape[-1]
    xn = (x - norm['mean_in'][..., :c1]) / norm['std_in'][..., :c1]
    yn = (y - norm['mean_tgt'][..., :c2]) / norm['std_tgt'][..., :c2]
    return xn, yn

stream = build_stream(data_dir_path)
print(f'stream: {len(stream)} steps')
meas = max(1, sum(not np.allclose(stream[0]['target'][..., i], 0.0) for i in range(3)))
print(f'measured channels (from first window): {meas}')

## 3. Forward passes (GPU): collect frozen CNO predictions + targets

In [ ]:
preds, tgts = [], []
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        mod.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    prev_t = prev_pair[1] if prev_pair else None
    pred_n, _ = mod.ttt_step(in_n, prev_t)
    prev_pair = (in_n, tg_n)
    preds.append(denorm(torch.as_tensor(pred_n).detach()).squeeze(0).cpu().numpy().astype(np.float32))
    tgts.append(tgt.squeeze(0).cpu().numpy().astype(np.float32))
pred_all = np.stack(preds, 0)[..., :2]
tgt_all = np.stack(tgts, 0)[..., :2]
print('pred/target (u,v only):', pred_all.shape)

## 4. Sweep the interval width

For each half-width `w` we set `lower = pred - w*|pred|`, `upper = pred + w*|pred|` (total width `2w*|pred|`) and score with the **official** SPS formula. `w=0.05` is the scorer's default. SPS scores are 0-100.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
import scoring as official  # official SPS (SIGMA_GLOBAL, aggregate_sps)

print(f"{'half-width':>12}{'total-width':>13}{'sps_score':>12}{'coverage':>10}{'mean_nil':>10}")
print('-' * 60)
rows = []
for w in [0.05, 0.075, 0.10, 0.15, 0.20, 0.30, 0.50]:
    lo = pred_all - w * np.abs(pred_all)
    hi = pred_all + w * np.abs(pred_all)
    sps, cov = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo, upper=hi)
    nil = ((hi - lo) / official.SIGMA_GLOBAL)
    row = (w, round(official.score_sps(sps), 2), round(cov, 4), round(float(np.mean(nil)), 3))
    rows.append(row)
    print(f"{w:>12.3f}{2*w:>13.3f}{row[1]:>12.2f}{row[2]:>10.4f}{row[3]:>10.3f}")

### How to read this
- **coverage** = fraction of graded elements whose target falls inside the band. This is what widening BUYS.
- **mean_nil** = average normalized interval width `(upper-lower)/SIGMA_GLOBAL`. This is what widening COSTS (`exp(-nil)` sharpness term).
- **sps_score** = overall (0-100). The width that maximizes it is the sweet spot; if all wider bands score higher than `w=0.05`, the default is too narrow (given this CNO predictor's accuracy).

> These are local SPS numbers on the subset you staged / attached — **not leaderboard**. They answer the directional question (narrow vs wide) for the frozen CNO predictor.

## 5. (Optional) Same sweep for the adaptive interval (submission_v5)

Repeated here with submission_v5's own online calibration (v4/v5 already return bounds on every step). Edit `VARIANT` to compare v4 (absolute) vs v5 (relative).

In [ ]:
VARIANT = 'submission_v5'
import importlib, shutil
vsub = REPO / 'submissions' / VARIANT
if not (vsub / 'model.pth').exists() and (sub_dir / 'model.pth').exists():
    shutil.copy(sub_dir / 'model.pth', vsub / 'model.pth')
sys.path.insert(0, str(vsub))
import submission as V
mvas = V.get_ttt_model(str(vsub), device)

lo_all, hi_all = [], []
prev_pair = None
for s in stream:
    inp = torch.tensor(s['input'], dtype=torch.float32, device=device).unsqueeze(0)
    tgt = torch.tensor(s['target'], dtype=torch.float32, device=device).unsqueeze(0)
    if s['is_first']:
        mvas.reset_ttt_state(); prev_pair = None
    in_n, tg_n = preprocess(inp, tgt)
    prev_t = prev_pair[1] if prev_pair else None
    pred_n, info = mvas.ttt_step(in_n, prev_t)
    prev_pair = (in_n, tg_n)
    lo_all.append(denorm(torch.as_tensor(info['lower']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
    hi_all.append(denorm(torch.as_tensor(info['upper']).detach()).squeeze(0).cpu().numpy().astype(np.float32)[..., :2])
lo = np.stack(lo_all, 0); hi = np.stack(hi_all, 0)
sps, cov = official.aggregate_sps(pred_all, tgt_all, 2, lower=lo, upper=hi)
nil = ((hi - lo) / official.SIGMA_GLOBAL)
print(f'[{VARIANT}] adaptive interval on same preds: SPS {official.score_sps(sps):.2f}, '
      f'cov {cov:.4f}, mean_nil {float(np.mean(nil)):.3f}')